In [24]:
import pandas as pd

df = pd.read_csv("datasets/product_vendor_combinations.csv")
df

,product_name,vendor_name
0,freescout,freescout-help-desk
1,civetweb,CivetWeb
2,CPython,Python Software Foundation
3,GoAnywhere MFT,Fortra
4,Firefox,Mozilla
...,...,...
43815,haystack,deepset-ai
43816,Establishment Billing Management System,SourceCodester
43817,eWeLink Cloud Service,CoolKIt
43818,WANotifier,Unknown


In [25]:
import json
import pandas as pd

def extract_json_arrays(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    arrays = []
    start = None
    depth = 0
    in_string = False
    escaped = False

    for i, char in enumerate(text):

        if char == "\\" and in_string:
            escaped = not escaped
            continue

        if char == '"' and not escaped:
            in_string = not in_string

        escaped = False

        if in_string:
            continue

        if char == "[":
            if depth == 0:
                start = i
            depth += 1

        elif char == "]":
            depth -= 1

            if depth == 0 and start is not None:
                arrays.append((start, text[start:i + 1]))
                start = None

    return arrays


records = []
failed_blocks = []

for position, block in extract_json_arrays("datasets/response ai.txt"):
    try:
        data = json.loads(block)

        if isinstance(data, list):
            records.extend(data)

    except json.JSONDecodeError as e:
        failed_blocks.append({
            "position": position,
            "error": str(e),
            "content": block
        })

seen = set()
deduped = []
for r in records:
    key = (r.get("Original Vendor"), r.get("Original Product"))
    if key not in seen:
        seen.add(key)
        deduped.append(r)
records = deduped

df_final = pd.DataFrame(records)

# Save failed blocks for manual review
if failed_blocks:
    with open("datasets/failed_blocks.txt", "w", encoding="utf-8") as f:
        for i, failure in enumerate(failed_blocks, 1):
            f.write(f"===== FAILED BLOCK {i} =====\n")
            f.write(f"Position: {failure['position']}\n")
            f.write(f"Error: {failure['error']}\n\n")
            f.write(failure["content"])
            f.write("\n\n\n")
print(f"Loaded {len(df_final)} rows")
print(f"Found {len(failed_blocks)} failed blocks")

Loaded 43745 rows
Found 5 failed blocks


Manually check the remaining: all 5 blocks missed categories

In [26]:
# Create a set of vendor-product combinations from the original df
original_combinations = set(zip(df['vendor_name'], df['product_name']))

# Check if df_final combinations exist in original df
df_final['in_original_df'] = df_final.apply(
    lambda row: (row['Original Vendor'], row['Original Product']) in original_combinations,
    axis=1
)

# Summary
print(f"Total rows in df_final: {len(df_final)}")
print(f"Rows found in original df: {df_final['in_original_df'].sum()}")
print(f"Rows NOT found in original df: {(~df_final['in_original_df']).sum()}")

# identify duplicate vendor-product combos
combo_counts = df_final.groupby(['Original Vendor', 'Original Product']).size().reset_index(name='count')
duplicate_combos = combo_counts[combo_counts['count'] > 1].sort_values('count', ascending=False)

print(f"Unique duplicate combinations: {len(duplicate_combos)}")
print(f"Total rows that are duplicates: {duplicate_combos['count'].sum()}")

# Remove rows not found in original df
removed_count = (~df_final['in_original_df']).sum()
df_final = df_final[df_final['in_original_df']].reset_index(drop=True)
print(f"Removed {removed_count} rows not found in original df. New length: {len(df_final)}")

duplicate_combos.head(94)


Total rows in df_final: 43745
Rows found in original df: 42700
Rows NOT found in original df: 1045
Unique duplicate combinations: 0
Total rows that are duplicates: 0
Removed 1045 rows not found in original df. New length: 42700


,Original Vendor,Original Product,count


In [28]:
df_final.isna().sum()

Original Vendor         0
Original Product        0
Category_1              0
Category_2              0
Category_3              0
Category_4              0
Category_5              0
Product             42700
Product_Original    42700
Generic Category    42700
in_original_df          0
dtype: int64

In [29]:
df_final = df_final.drop(columns=['Product', 'Product_Original', 'Generic Category', 'in_original_df'])

In [31]:
df_final.to_csv("datasets/5 categories.csv", index=False)